# Stage 0 — Hypothesis v4: TSMOM + Cross-Asset Consensus Filter

**Date locked:** 2026-05-24  
**Author:** Anthony Chung  
**Version:** v4.0  
**Predecessors:**
- v1.0 Asian fade — FAILED Stage 2
- v2.0 Stretched VP fade — FAILED Stage 2
- v3.0 TSMOM (pure) — passed Stage 2+3, FAILED Stage 4 (53% profitable months)

**Status:** ACTIVE

---

## Why v4 (not v3 tweak)

v3 TSMOM showed clear evidence of edge (PF 2.1, Sharpe 1.3 OOS) but failed walk-forward because the edge concentrates in regime-trending months and disappears in choppy months.

v4 introduces a **fundamentally new mechanism**: a cross-asset consensus filter that only allows trading when the universe shows directional agreement. This is NOT a v3 parameter tweak — it's a new signal layer.

Per SOP, this is a fresh hypothesis with new Stage 0 lockdown.

## Thesis

Time-series momentum on individual coins (v3) generates an edge during clear-trend regimes but loses to noise during choppy regimes. A cross-asset consensus filter — only acting when ≥N of 6 coins agree on direction — should preserve the trend-regime edge while sitting out the chop.

## Market Inefficiency Targeted

**Regime-conditional momentum with consensus filter**:

1. v3 confirmed that TSMOM has genuine edge on crypto (academic literature + our IS/OOS results).
2. v3 walk-forward exposed that the edge is concentrated in months when crypto markets exhibit broad directional consensus.
3. Choppy months — where some coins trend up and others trend down with similar magnitudes — produced losses by whipsawing both sides.
4. A consensus filter is a structural signal that the macro crypto regime IS trending: when 5 of 6 coins all favor LONG, that's a different market state than when 3 favor LONG and 3 favor SHORT.
5. Sitting out chop regimes saves whipsaw losses; participating in consensus regimes captures the trend edge.

## Theoretical Support

| Reference | Contribution |
|-----------|--------------|
| Han, Kang, Ryu (2023) | TSMOM works on crypto — base layer of this strategy |
| Moskowitz, Ooi, Pedersen (2012) | Cross-asset TSMOM seminal paper; combined-asset signals |
| Asness, Moskowitz, Pedersen (2013) — "Value and Momentum Everywhere" | Cross-asset confirmation of momentum across markets |
| Hurst, Ooi, Pedersen (2017) — "A Century of Evidence on Trend-Following Investing" | Trend-following works better when regime trends; suggests regime filter |

The cross-asset consensus filter is novel for crypto specifically but follows from the multi-asset TSMOM literature where market-wide trending is a recognized regime indicator.

## Strategy Specification

### Universe
- Same 6 coins as v3: BTC, ETH, SOL, AVAX, XRP, BNB

### Per-coin signal (same as v3 TSMOM)
```
trailing_return = price[now] / price[now - lookback_weeks × 168h] - 1
per_coin_signal = LONG if trailing_return > +signal_threshold
                  SHORT if trailing_return < -signal_threshold
                  FLAT otherwise
```

### NEW — Cross-asset consensus filter
At each rebalance, count:
```
n_long  = number of coins with LONG signal
n_short = number of coins with SHORT signal
max_consensus = max(n_long, n_short)

if max_consensus >= consensus_threshold:
    take all per-coin signals (LONG/SHORT/FLAT) as-is
else:
    override all positions to FLAT  # no clear regime, sit out
```

### Execution
Identical to v3: weekly rebalance Monday 00:00 UTC, hold 1 week, HORROR cost.

### Sizing
Equal weight 1/6 per coin slot; FLAT positions hold cash.

## Locked Parameter Search Space

| Parameter | Search values | Notes |
|-----------|---------------|-------|
| `lookback_weeks` | [2, 4, 8] | v3 found 4w optimal; broader exploration here |
| `signal_threshold` | [0.00, 0.02, 0.05] | dead-zone width |
| `consensus_threshold` | [3, 4, 5] | min coins agreeing; 3=loose / 5=strict |
| `vol_target` | [False] | locked — v3 showed marginal benefit only |

**Total trials:** 3 × 3 × 3 = **27**

Smaller grid than v3 (30) because adding consensus is the focus; vol_target locked off.

## Falsifiable Predictions

| # | Test | Threshold | Stage |
|---|------|-----------|-------|
| 1 | At least 1 of 27 IS trials has PF > 1.0 | required | 2 |
| 2 | OOS-1 PF retention | ≥ 70% | 3 |
| 3 | **Walk-forward profitable months** (the v3 killer) | **≥ 80%** | 4 |
| 4 | Holdout PF | > 1.3 | 5 |
| 5 | Holdout Sharpe | > 1.0 | 5 |
| 6 | Holdout MaxDD | < 30% | 5 |
| 7 | Deflated Sharpe (N=27) | > 0 | 5 |

**Crucial test:** Stage 4 — if consensus filter doesn't dramatically improve walk-forward consistency over v3 (which had 53%), the filter is not adding value and v4 dies. The whole point of v4 is to address v3's Stage 4 failure.

**Secondary structural prediction:** higher consensus_threshold should produce fewer but more concentrated trades, with higher per-trade PF but lower n_trades.

## Pre-Commitment Statement

> I have not examined any v4 backtest prior to this commit. The parameter ranges for v4 are independent of v3 results — I am explicitly not re-using v3's optimal config (4-week, 2% threshold) as a tuned starting point. The grid here is independent.
>
> The consensus filter mechanism is novel to v4 and was not part of v3.
>
> If v4 fails at any stage, no v5 may attempt 'just one more parameter' on TSMOM-family. v5 (if ever) requires a fundamentally different mechanism (e.g., pairs trading, breakout, calendar, on-chain).
>
> — Anthony Chung, 2026-05-24